<a href="https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [6]:
import duckdb
import pandas as pd

from google.colab import userdata
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("Add HF_TOKEN in Colab Secrets and enable Notebook access.")

con = duckdb.connect()
con.execute(
    "CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{hf_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FEB = f"{REL}/fact_content_daily_performance/month=2026-02/*.parquet"
MAR = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

feature_sql = f"""
WITH february_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS feb_impressions,
        AVG(NULLIF(gsc_avg_position, 0)) AS feb_avg_position,
        SUM(
    CASE
        WHEN ga4_data_available IS TRUE
        THEN scroll_events
    END
) AS feb_scroll_events,
        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN sessions_social
            END
        ) AS feb_sessions_social,
        COUNT(*) AS feb_observed_days,
        MAX(
            CASE
                WHEN ga4_data_available IS TRUE THEN 1
                ELSE 0
            END
        ) AS has_ga4_data
    FROM read_parquet('{FEB}')
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
),
march_outcomes AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks)::DOUBLE
            / NULLIF(SUM(gsc_impressions), 0) AS march_ctr,
        AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position
    FROM read_parquet('{MAR}')
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING
        SUM(gsc_impressions) >= 100
        AND AVG(NULLIF(gsc_avg_position, 0)) IS NOT NULL
),
bucketed_outcomes AS (
    SELECT *,
        CASE
            WHEN march_avg_position <= 3 THEN '1-3'
            WHEN march_avg_position <= 10 THEN '4-10'
            WHEN march_avg_position <= 20 THEN '11-20'
            WHEN march_avg_position <= 50 THEN '21-50'
            ELSE '51+'
        END AS march_position_bucket
    FROM march_outcomes
),
labeled_outcomes AS (
    SELECT *,
        MEDIAN(march_ctr) OVER (
            PARTITION BY march_position_bucket
        ) AS median_march_ctr_by_bucket
    FROM bucketed_outcomes
)
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.feb_impressions,
    f.feb_avg_position,
    f.feb_scroll_events,
    f.feb_sessions_social,
    f.feb_observed_days,
    f.has_ga4_data,
    CASE
        WHEN f.feb_avg_position <= 3 THEN '1-3'
        WHEN f.feb_avg_position <= 10 THEN '4-10'
        WHEN f.feb_avg_position <= 20 THEN '11-20'
        WHEN f.feb_avg_position <= 50 THEN '21-50'
        ELSE 'missing_or_51+'
    END AS feb_position_bucket,
    CASE
        WHEN o.march_ctr < o.median_march_ctr_by_bucket THEN 1
        ELSE 0
    END AS is_march_below_bucket_median_ctr
FROM february_features AS f
INNER JOIN labeled_outcomes AS o
    USING (client_hash_id, content_hash_id)
"""

feature_frame = con.execute(feature_sql).df()

numeric_features = [
    "feb_impressions",
    "feb_avg_position",
    "feb_scroll_events",
    "feb_sessions_social",
    "feb_observed_days",
]

categorical_features = [
    "feb_position_bucket",
    "has_ga4_data",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("impute", SimpleImputer(strategy="median", add_indicator=True))
                ]
            ),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

X = preprocessor.fit_transform(feature_frame)
y = feature_frame["is_march_below_bucket_median_ctr"]

print("=== Feature Vector Check ===")
print("Feature-frame rows:", len(feature_frame))
print("Encoded feature-matrix shape:", X.shape)
print("March target positive rate:", round(y.mean(), 3))

display(feature_frame.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Feature Vector Check ===
Feature-frame rows: 86560
Encoded feature-matrix shape: (86560, 15)
March target positive rate: 0.385


,client_hash_id,content_hash_id,feb_impressions,feb_avg_position,feb_scroll_events,feb_sessions_social,feb_observed_days,has_ga4_data,feb_position_bucket,is_march_below_bucket_median_ctr
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,13.425718,NaN,NaN,28,0,11-20,1
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.495085,0.0,0.0,28,1,4-10,1
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,10.490023,0.0,0.0,28,1,11-20,1
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,38.436254,1.0,0.0,28,1,21-50,0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,9.710810,1.0,0.0,28,1,4-10,0
5,client_e547b89c05043229,content_a2bd730a7cf68316,551.0,6.017373,0.0,0.0,28,1,4-10,1
6,client_e547b89c05043229,content_cbe43d4b6ce2d320,291.0,5.055301,0.0,0.0,28,1,4-10,0
7,client_e547b89c05043229,content_babd931911c9ee33,2680.0,5.046562,5.0,0.0,28,1,4-10,0
8,client_e547b89c05043229,content_9c36ace83c73b5eb,416.0,40.800604,0.0,0.0,28,1,21-50,0
9,client_e547b89c05043229,content_431784c057b25a5d,3641.0,8.784133,0.0,0.0,28,1,4-10,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

| Feature | Meaning | Missing-value handling | Categorical? | Available when? |
|---|---|---|---|---|
| `feb_impressions` | Total February Search Console impressions | Median imputation if missing, plus a missingness indicator | No | At the end of February, before the March outcome window |
| `feb_avg_position` | Average February search position | Median imputation if missing, plus a missingness indicator | No | At the end of February |
| `feb_scroll_events` | February scroll events when GA4 data is available | Missing when GA4 is unavailable; median imputation plus a missingness indicator | No | At the end of February |
| `feb_sessions_social` | February social sessions when GA4 data is available | Missing when GA4 is unavailable; median imputation plus a missingness indicator | No | At the end of February |
| `feb_observed_days` | Number of eligible February daily records | Median imputation if missing, plus a missingness indicator | No | At the end of February |
| `feb_position_bucket` | Bucket created from February average position | Most-frequent-category imputation, then one-hot encoding | Yes | At the end of February |
| `has_ga4_data` | Whether any eligible February GA4 data exists | No fill; treated as a category and one-hot encoded | Yes | At the end of February |

The March label is not a feature. It is measured after February and is used only as the outcome for the leakage check.

In [7]:
feature_columns_for_check = (
    numeric_features + categorical_features
)

feature_check = pd.DataFrame({
    "feature": feature_columns_for_check,
    "dtype": [
        str(feature_frame[column].dtype)
        for column in feature_columns_for_check
    ],
    "missing_values": [
        int(feature_frame[column].isna().sum())
        for column in feature_columns_for_check
    ],
    "missing_pct": [
        round(feature_frame[column].isna().mean() * 100, 2)
        for column in feature_columns_for_check
    ],
})

print("=== Feature Notes: Real Missingness Check ===")
display(feature_check)

print("\n=== Categorical Values ===")
for column in categorical_features:
    print(
        f"{column}:",
        sorted(feature_frame[column].dropna().unique().tolist())
    )

print("\n=== Encoding Check ===")
print("Encoded matrix shape:", X.shape)
print(
    "Numeric features use median imputation with missingness indicators."
)
print(
    "Categorical features use most-frequent imputation and one-hot encoding."
)

=== Feature Notes: Real Missingness Check ===


,feature,dtype,missing_values,missing_pct
0,feb_impressions,float64,0,0.00
1,feb_avg_position,float64,75,0.09
2,feb_scroll_events,float64,66850,77.23
3,feb_sessions_social,float64,66850,77.23
4,feb_observed_days,int64,0,0.00
5,feb_position_bucket,object,0,0.00
6,has_ga4_data,int32,0,0.00



=== Categorical Values ===
feb_position_bucket: ['1-3', '11-20', '21-50', '4-10', 'missing_or_51+']
has_ga4_data: [0, 1]

=== Encoding Check ===
Encoded matrix shape: (86560, 15)
Numeric features use median imputation with missingness indicators.
Categorical features use most-frequent imputation and one-hot encoding.


### Leakage attack plan

**Timeline:** February features → March label. Every allowed feature is calculated only from February records, before the March outcome window.

**Label-derived attack:** I will deliberately add a copy of the March label to the feature vector. The leaked model should show an unrealistically high score. I will then remove that column and retain only the honest grouped-holdout result.

**Future-window attack:** no feature name or input may contain March outcomes, CTR, clicks, the label, a future window, or a pre-existing decision score.

**Product-flag attack:** no product flag, health score, priority score, action type, or similar decision-derived field is allowed in the feature vector.

**Validation:** the comparison uses a client-grouped holdout, so the model is evaluated on clients not seen during training.

In [8]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

safe_feature_names = numeric_features + categorical_features

suspect_terms = [
    "march",
    "label",
    "ctr",
    "click",
    "target",
    "future",
    "next",
    "priority",
    "health",
    "action",
    "flag",
]

suspect_features = [
    feature
    for feature in safe_feature_names
    if any(term in feature.lower() for term in suspect_terms)
]

print("=== Leakage Name Audit ===")
print("Safe feature names:", safe_feature_names)
print("Suspect feature names found:", suspect_features)

if suspect_features:
    raise ValueError("Unsafe feature names found. Stop and remove them.")
else:
    print("Result: no label, future-window, or product-decision fields in the safe vector.")

X_safe = feature_frame[safe_feature_names].copy()
y = feature_frame["is_march_below_bucket_median_ctr"].copy()
groups = feature_frame["client_hash_id"].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_index, test_index = next(
    splitter.split(X_safe, y, groups=groups)
)

X_train = X_safe.iloc[train_index]
X_test = X_safe.iloc[test_index]
y_train = y.iloc[train_index]
y_test = y.iloc[test_index]

honest_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "model",
            DecisionTreeClassifier(
                max_depth=5,
                random_state=42,
            ),
        ),
    ]
)

honest_pipeline.fit(X_train, y_train)

honest_auc = roc_auc_score(
    y_test,
    honest_pipeline.predict_proba(X_test)[:, 1],
)

# Deliberately add the answer itself: this is leakage.
X_leaked = X_safe.copy()
X_leaked["leaked_march_label_copy"] = y

leaky_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "impute",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True,
                        ),
                    )
                ]
            ),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "impute",
                        SimpleImputer(strategy="most_frequent"),
                    ),
                    (
                        "one_hot",
                        OneHotEncoder(handle_unknown="ignore"),
                    ),
                ]
            ),
            categorical_features,
        ),
        (
            "deliberate_leak",
            "passthrough",
            ["leaked_march_label_copy"],
        ),
    ]
)

leaky_pipeline = Pipeline(
    steps=[
        ("preprocessor", leaky_preprocessor),
        (
            "model",
            DecisionTreeClassifier(
                max_depth=5,
                random_state=42,
            ),
        ),
    ]
)

leaky_pipeline.fit(
    X_leaked.iloc[train_index],
    y_train,
)

leaked_auc = roc_auc_score(
    y_test,
    leaky_pipeline.predict_proba(
        X_leaked.iloc[test_index]
    )[:, 1],
)

print("\n=== Grouped-holdout leakage experiment ===")
print("Held-out client count:", groups.iloc[test_index].nunique())
print("Test-set base rate:", round(y_test.mean(), 3))
print("Honest ROC-AUC:", round(honest_auc, 3))
print("Leaked ROC-AUC:", round(leaked_auc, 3))

del X_leaked["leaked_march_label_copy"]
print(
    "Removed leaked_march_label_copy. "
    "Only the honest feature vector remains."
)

=== Leakage Name Audit ===
Safe feature names: ['feb_impressions', 'feb_avg_position', 'feb_scroll_events', 'feb_sessions_social', 'feb_observed_days', 'feb_position_bucket', 'has_ga4_data']
Suspect feature names found: []
Result: no label, future-window, or product-decision fields in the safe vector.

=== Grouped-holdout leakage experiment ===
Held-out client count: 10
Test-set base rate: 0.451
Honest ROC-AUC: 0.61
Leaked ROC-AUC: 1.0
Removed leaked_march_label_copy. Only the honest feature vector remains.


### Excluded fields

- **`client_hash_id` and `content_hash_id`** — used only for grouping and the client-holdout split; never model features.
- **March clicks, March CTR, position-bucket median CTR, and the March label** — excluded because they define the March outcome and would leak the answer into February features.
- **June 2026 and later partitions** — excluded because they are future data relative to the February-to-March experiment.
- **Product decision fields, priority scores, health scores, action types, and flags** — excluded because they would encode an existing decision rather than an observed signal.
- **Raw client names, URLs, titles, and private queries** — excluded for public safety; this notebook uses only pseudonymized IDs and aggregated metrics.
- **GA4-unavailable scroll and social values** — not converted into zero; they remain missing and are handled with imputation plus a missingness indicator.

In [9]:
forbidden_feature_names = {
    "client_hash_id",
    "content_hash_id",
    "march_impressions",
    "march_ctr",
    "march_avg_position",
    "median_march_ctr_by_bucket",
    "is_march_below_bucket_median_ctr",
    "gsc_clicks",
    "health_score",
    "priority_score",
    "action_type",
    "product_flag",
    "url",
    "query",
    "title",
    "client_name",
}

used_model_features = set(safe_feature_names)

forbidden_in_model = sorted(
    used_model_features.intersection(forbidden_feature_names)
)

private_name_terms = [
    "url",
    "query",
    "title",
    "client_name",
    "domain",
]

private_like_features = [
    feature
    for feature in safe_feature_names
    if any(term in feature.lower() for term in private_name_terms)
]

ga4_unavailable = feature_frame[
    feature_frame["has_ga4_data"] == 0
]

ga4_values_present_when_unavailable = (
    ga4_unavailable[
        ["feb_scroll_events", "feb_sessions_social"]
    ]
    .notna()
    .sum()
    .sum()
)

print("=== Exclusions and Privacy Check ===")
print("Forbidden fields used as model features:", forbidden_in_model)
print("Private-like feature names found:", private_like_features)
print("Rows without eligible February GA4 data:", len(ga4_unavailable))
print(
    "GA4 values present when GA4 was unavailable:",
    int(ga4_values_present_when_unavailable),
)

if forbidden_in_model:
    raise ValueError("Forbidden target, ID, or decision fields found.")

if private_like_features:
    raise ValueError("Private-like fields found in model features.")

if ga4_values_present_when_unavailable != 0:
    raise ValueError(
        "GA4-unavailable values were incorrectly treated as observed data."
    )

print(
    "\nResult: the feature vector excludes IDs, March outcomes, "
    "product decisions, and private-like fields."
)

=== Exclusions and Privacy Check ===
Forbidden fields used as model features: []
Private-like feature names found: []
Rows without eligible February GA4 data: 66850
GA4 values present when GA4 was unavailable: 0

Result: the feature vector excludes IDs, March outcomes, product decisions, and private-like fields.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.